# Part 2：从零实现 BPE Tokenizer

> **前情回顾**：我们在 Part 1 学了字符级 tokenizer（太碎）和词级 tokenizer（遇生词崩溃），需要一个折中方案 —— 子词级。
>
> 本 Part 目标：**手写一个完整的 BPE tokenizer，每一步合并都打印出来给你看。**

## BPE 一句话总结

**反复找出现最频繁的相邻 token 对，把它们合并成一个新 token。**

重复 N 次，词表会自动「生长」出有意义的子词。

### 0. 重新加载语料（和 Part 1 一样）

In [ ]:
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog",
    "i love my cat",
    "i love my dog",
    "the cat is cute",
    "the dog is happy",
    "the mat is soft",
    "the log is hard",
    "cats and dogs are friends",
]

print("我们的语料：")
for i, s in enumerate(corpus):
    print(f"  [{i}] {s}")

### 1. 热身：BPE 的第一步 — 从哪里开始？

BPE 的起点是**字符级**。先把所有文本拆成字符，每个字符一个 token。

但有个细节：怎么标记「单词之间的空格」？

在实际的 BPE（比如 GPT 用的）中，空格不会被当普通字符合并掉。
但为了教学清晰，我们的 mini 版**先把空格也当普通字符处理** —— 这样你能清楚看到 BPE 如何把字符拼成子词。

In [ ]:
# 第一步：把所有句子拆成字符列表
# 注意：空格 ' ' 就是一个普通字符

def text_to_tokens(text):
    """把一段文本拆成字符级 token 列表"""
    return list(text)

# 看第一条句子
sentence = corpus[0]
chars = text_to_tokens(sentence)

print(f"原文:   '{sentence}'")
print(f"拆成字符: {chars}")
print(f"共 {len(chars)} 个 token")

# 把整条语料的每条句子都拆成字符列表
corpus_tokens = [text_to_tokens(s) for s in corpus]
print(f"\n语料中每条句子的字符数: {[len(t) for t in corpus_tokens]}")

### 2. BPE 核心操作一：统计相邻 pair 的频率

给定一个 token 序列（比如 `['t','h','e',' ','c','a','t']`），我们找**所有相邻的两个 token**，数它们各自出现了多少次。

```
['t', 'h', 'e', ' ', 'c', 'a', 't']
  ↑──↑            
 ('t','h'): 1次
      ↑──↑
     ('h','e'): 1次
          ↑──↑
         ('e',' '): 1次
              ↑──↑
             (' ','c'): 1次
                  ↑──↑
                 ('c','a'): 1次
                      ↑──↑
                     ('a','t'): 1次
```

但我们统计的是**整个语料**中的所有相邻 pair，找出全局最高频的那个。

In [ ]:
from collections import defaultdict

def count_pairs(token_lists):
    """
    统计整个语料中所有相邻 token pair 的出现频率
    
    参数:
        token_lists: List[List[str]]，每条句子是一个字符列表
    返回:
        dict: {(token_a, token_b): 出现次数}
    """
    pairs = defaultdict(int)
    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            pairs[pair] += 1
    return dict(pairs)

# 统计初始状态（全字符级）的 pair 频率
initial_pairs = count_pairs(corpus_tokens)

print("=== 初始状态下所有相邻 pair 的频率 ===")
# 按频率从高到低排序打印
for pair, count in sorted(initial_pairs.items(), key=lambda x: -x[1]):
    print(f"  {pair}: {count} 次")

print(f"\n最高频 pair: {max(initial_pairs, key=initial_pairs.get)}")
print(f"出现了 {initial_pairs[max(initial_pairs, key=initial_pairs.get)]} 次")

### 3. BPE 核心操作二：合并最高频的 pair

找到了最高频 pair，下一步就是把语料中**所有这个 pair 的两个连续 token 合并成一个**。

```
合并前: ['t', 'h', 'e', ' ', 'c', 'a', 't']
最高频 pair: ('t', 'h')
合并后: ['th', 'e', ' ', 'c', 'a', 't']
            ↑  't'和'h'被压缩成了一个 token 'th'
```

In [ ]:
def merge_pair(token_lists, pair_to_merge):
    """
    把语料中所有出现的指定 pair 合并成一个 token
    
    参数:
        token_lists: List[List[str]]，当前语料的 token 表示
        pair_to_merge: (str, str)，要合并的 pair
    返回:
        合并后的新 token_lists
    """
    a, b = pair_to_merge
    new_token = a + b  # 新 token 就是两个字符拼接
    
    new_token_lists = []
    for tokens in token_lists:
        new_tokens = []
        i = 0
        while i < len(tokens):
            # 如果当前位置和下一个位置正好是这个 pair，合并！
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(new_token)
                i += 2  # 跳两步
            else:
                new_tokens.append(tokens[i])
                i += 1
        new_token_lists.append(new_tokens)
    
    return new_token_lists, new_token

# 演示：做一次合并
best_pair = max(initial_pairs, key=initial_pairs.get)
print(f"要合并的 pair: {best_pair}")

merged_tokens, new_token = merge_pair(corpus_tokens, best_pair)
print(f"新 token: '{new_token}'")
print(f"\n合并后每条句子的 token 序列：")
for i, tokens in enumerate(merged_tokens):
    print(f"  [{i}] {tokens}")

### 4. 观察：一次合并改变了什么？

注意看上面的输出 —— 原来分散的 `'t','h'` 被合并成了 `'th'`。

但这只是第一步。BPE 的核心是**反复做这件事**：
1. 统计 pair 频率
2. 找最高频 pair
3. 合并
4. 记录这次合并（这叫一个 **merge rule**）
5. 回到第 1 步

每次合并都会产生新的相邻 pair（比如 `'th'+'e'` → `'the'`），所以这个过程会逐渐长出越来越长的 token。

下面我们完整跑一遍！

### 5. 完整的 BPE 训练过程

**关键参数**：`num_merges` —— 想做多少次合并？

- `num_merges=10` → 最终词表 ≈ 初始字符数 + 10 = 约 30 个 token
- `num_merges=50` → 最终词表 ≈ 70 个 token
- GPT-3.5 的 `num_merges` ≈ 10 万次

我们先设 `num_merges=15`，每一步都打印出来，看词表怎么「生长」。

In [ ]:
class BPETokenizer:
    """
    BPE Tokenizer 完整实现
    
    三个核心功能：
    1. train()  — 从语料学习 merge rules
    2. encode() — 文本 → token ID 列表
    3. decode() — token ID 列表 → 文本
    """
    
    def __init__(self):
        # merge_rules: 按顺序记录每次合并的 pair
        # 这是 BPE 最核心的数据结构，encode 时按这个顺序贪心应用
        self.merge_rules = []
        # 最终词表
        self.vocab = set()
        self.stoi = {}
        self.itos = {}
    
    def train(self, texts, num_merges=15, verbose=True):
        """
        BPE 训练
        
        参数:
            texts: 语料列表
            num_merges: 合并次数
            verbose: 是否打印每一步
        """
        # Step 0: 初始化为字符级
        token_lists = [list(text) for text in texts]
        
        if verbose:
            print(f"{'='*60}")
            print(f"BPE 训练开始！初始状态: 每条句子按字符拆分")
            print(f"{'='*60}")
            print(f"初始字符集大小: {len(set(c for t in token_lists for c in t))}")
            print()
        
        for step in range(num_merges):
            # Step 1: 统计所有 pair 的频率
            pairs = defaultdict(int)
            for tokens in token_lists:
                for i in range(len(tokens) - 1):
                    pairs[(tokens[i], tokens[i + 1])] += 1
            
            if not pairs:
                break
            
            # Step 2: 找最高频 pair
            best_pair = max(pairs, key=pairs.get)
            best_count = pairs[best_pair]
            
            # Step 3: 记录 merge rule
            self.merge_rules.append(best_pair)
            
            # Step 4: 合并
            a, b = best_pair
            new_token = a + b
            
            new_token_lists = []
            for tokens in token_lists:
                new_tokens = []
                i = 0
                while i < len(tokens):
                    if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                        new_tokens.append(new_token)
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                new_token_lists.append(new_tokens)
            
            token_lists = new_token_lists
            
            if verbose:
                current_vocab = set(t for tokens in token_lists for t in tokens)
                print(f"Step {step+1:2d}: merge {best_pair} → '{new_token}'  (出现 {best_count} 次) | 当前词表大小: {len(current_vocab)}")
        
        # 建立最终词表
        self.vocab = set(t for tokens in token_lists for t in tokens)
        sorted_vocab = sorted(list(self.vocab))
        self.stoi = {t: i for i, t in enumerate(sorted_vocab)}
        self.itos = {i: t for i, t in enumerate(sorted_vocab)}
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"训练完成！")
            print(f"  - 合并次数: {len(self.merge_rules)}")
            print(f"  - 最终词表大小: {len(self.vocab)} 个 token")
            print(f"  - Merge rules: {self.merge_rules}")
            print(f"{'='*60}")
        
        return token_lists


In [ ]:
# 训练 BPE tokenizer！设 15 步合并，每一步都打印
bpe = BPETokenizer()
final_tokens = bpe.train(corpus, num_merges=15, verbose=True)

### 6. 看懂训练日志

回头仔细看上面的输出 —— 你会发现一个规律：

```
Step 1: 合并空格 + 后面的字母（比如 ' t'）      ← 空格是最多的字符
Step 2: 合并更多空格+字母的组合
Step 3: 'th' 出现，因为 "the" 出现了很多次
Step 4: 'he' 出现
Step 5: 'th'+'e' → 'the'  整个词诞生了！
...
```

**关键洞察**：BPE 不是一次性把 "the" 识别为一个词的，而是：
- 先看到 `t+h` 出现很多次 → 合并成 `th`
- 然后发现 `th+e` 也出现很多次 → 合并成 `the`
- 这是一个**逐渐生长**的过程

这也是为什么词表大小可控：你可以说「我就做 15 次合并」，得到的词表就是初始字符 + 15 个新 token。

### 7. BPE 的 encode：如何用 merge rules 编码？

现在有了 merge rules，怎么把新文本编码成 token？

**思路**：
1. 先把文本拆成字符
2. 按训练时的 merge rules 顺序，贪心地应用每一次合并
3. 最后剩下的 token 去词表查 ID

```
"the cat"
  → ['t','h','e',' ','c','a','t']
  → 应用 rule 1: (' ','t') → ' t'  
  → 应用 rule 2: (' ','c') → ' c'
  → 应用 rule 3: ('t','h') → 'th'
  → 应用 rule 4: ('h','e') → 'he'
  → 应用 rule 5: ('th','he') → 'the'
  ...
  → ['the', ' c', 'a', 't']
```

**这个顺序很重要！** 必须和训练时的 merge 顺序一致，先合并的先检查。

In [ ]:
# 给 BPETokenizer 加上 encode 方法
def bpe_encode(self, text):
    """
    BPE 编码：文本 → token ID 列表
    
    核心逻辑：按训练时的 merge rules 顺序，贪心地合并
    """
    # Step 1: 拆成字符
    tokens = list(text)
    
    # Step 2: 依次应用每条 merge rule
    for (a, b) in self.merge_rules:
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and tokens[i] == a and tokens[i + 1] == b:
                new_tokens.append(a + b)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    # Step 3: 每个 token 去词表查 ID
    ids = []
    for token in tokens:
        if token in self.stoi:
            ids.append(self.stoi[token])
        else:
            # 没见过的 token → 拆回字符（这就是 BPE 不崩的原因！）
            for ch in token:
                ids.append(self.stoi[ch])
    return ids

# 把方法挂上去
BPETokenizer.encode = bpe_encode

print("encode 方法已添加！")

In [ ]:
# 测试 encode
text = "the cat"
ids = bpe.encode(text)
print(f"原文: '{text}'")
print(f"Token IDs: {ids}")
print(f"Token 数量: {len(ids)}")

# 看看每个 ID 对应什么 token
print(f"\n逐个解释:")
for i, tid in enumerate(ids):
    print(f"  ID {tid} → '{bpe.itos[tid]}'")

# 对比字符级
print(f"\n对比: 原文字符数 = {len(text)}，BPE token 数 = {len(ids)}，压缩比 = {len(text)/len(ids):.1f}x")

### 8. BPE 的 decode：最简单的一步

decode 比 encode 简单太多了 —— 就是把每个 token ID 查表变回字符串，然后拼起来。

In [ ]:
def bpe_decode(self, ids):
    """
    BPE 解码：token ID 列表 → 文本
    就是查表 + 拼接，没有任何其他操作
    """
    return "".join([self.itos[i] for i in ids])

BPETokenizer.decode = bpe_decode

# 测试完整编解码
for test_text in ["the cat", "my dog is happy", "i love cats"]:
    ids = bpe.encode(test_text)
    recovered = bpe.decode(ids)
    status = "✅" if test_text == recovered else "❌"
    print(f"{status} '{test_text}' → {ids} → '{recovered}'")

### 9. BPE 解决 OOV 问题的关键：回头看 encode 的最后一步

在 encode 方法里，有一行关键的代码：

```python
if token in self.stoi:
    ids.append(self.stoi[token])
else:
    for ch in token:       # ← 关键！
        ids.append(self.stoi[ch])  # 拆回字符
```

**这就是 BPE 不会像词级 tokenizer 那样崩溃的终极原因。**

遇到完全没见过的词，BPE 会退回到字符级编码 —— 反正字符一定在词表里。

而一个好的 BPE tokenizer，经过足够多次 merge 后，绝大多数常见词和子词都已经在词表里了，很少需要退回到字符级。

In [ ]:
# 演示：编码一个语料中不存在的句子
unseen = "a zebra runs fast"
print(f"测试生僻句: '{unseen}'")
print(f"这个词没有出现在训练语料中！")

ids = bpe.encode(unseen)
recovered = bpe.decode(ids)

print(f"\nToken IDs: {ids}")
print(f"解码回来: '{recovered}'")
print(f"状态: {'✅ 没崩！' if unseen == recovered else '⚠️'}")

# 看看 BPE 把它拆成了什么
print(f"\n逐个 token:")
for tid in ids:
    print(f"  '{bpe.itos[tid]}'", end="")
print()

### 10. 一个小实验：不同 merge 次数的影响

merge 次数越多，词表越大，token 越「整」。我们来对比一下。

In [ ]:
# 用不同 merge 次数训练多个 tokenizer
for n_merges in [5, 15, 30]:
    t = BPETokenizer()
    t.train(corpus, num_merges=n_merges, verbose=False)
    
    test = "the cat sat on the mat"
    ids = t.encode(test)
    
    print(f"merge={n_merges:2d} | 词表={len(t.vocab):2d} | token数={len(ids):2d} | tokens: {[t.itos[i] for i in ids]}")

print()
print("观察：merge 次数越多，'the' 等常见词越可能作为整体保留，token 数越少。")

### 11. 现实中的 BPE 和我们实现的区别

我们的 mini 版为了教学清晰做了简化。工业级 BPE（如 GPT-2/3/4 用的）会加这些优化：

| 我们的版本 | 工业版 |
|-----------|--------|
| 从字符开始 | 从 **bytes** 开始（256 个字节，能处理所有 Unicode） |
| 空格当普通字符 | 用特殊符号 `Ġ` 标记单词开头 |
| 全量统计频率 | 用优先队列加速 |
| 10 条句子 | 数十亿词 |
| 纯 Python | C++/Rust |

但**核心原理一模一样**：统计 pair 频率 → 合并最高频 → 重复。

你现在已经能看懂 GPT-2 的 `encoder.json` 和 `vocab.bpe` 里每一条 merge rule 是什么意思了。

### 12. 特殊符号（Special Tokens）是哪来的？

你可能见过这些符号：`<BOS>` `<EOS>` `<PAD>` `<UNK>` `<SEP>` `<CLS>` `<MASK>`。
它们**不是 BPE 训练出来的**，而是**人工手动加到词表里的**。

```
BPE 训练完的词表:  {"a":0, "b":1, ..., "the":499, ...}
人工追加特殊符号:  {"a":0, "b":1, ..., "the":499, "<PAD>":500, "<BOS>":501, "<EOS>":502, "<UNK>":503}
```

**为什么需要它们？** 因为神经网络需要知道「句子在哪开始、在哪结束、遇到不认识的东西怎么办」。

| 符号 | 全称 | 作用 | 谁在用 |
|:---|:---|:---|:---|
| `<BOS>` | Beginning of Sequence | 标记句子开头 | GPT 系列（隐式）、翻译模型 |
| `<EOS>` | End of Sequence | 标记句子结束，**生成时遇到它就停** | 几乎所有 LLM |
| `<PAD>` | Padding | 把不等长的句子补成等长（batch 训练必须） | 所有需要 batch 训练的模型 |
| `<UNK>` | Unknown | 遇到词表里没有的词，统一映射成它 | 老式模型（BERT 之前） |
| `<SEP>` | Separator | 分隔两个句子（如问答对） | BERT |
| `<CLS>` | Classification | 放在句子开头，它的输出向量代表整句含义 | BERT |
| `<MASK>` | Mask | 盖住一个词让模型猜（MLM 训练） | BERT |

**关键理解**：
- BPE 本身不会产生这些符号——它们不是从文本中「学」出来的
- 它们是**训练前手动插入**的，就像给词表「预留座位」
- 不同模型用不同的特殊符号集合，GPT 只用 `<EOS>`，BERT 用 `<CLS>` `<SEP>` `<MASK>` `<PAD>`

In [ ]:
# 演示：给 BPE 词表手动添加特殊符号
print("=== 特殊符号演示 ===")
print()

special_tokens = ["<PAD>", "<BOS>", "<EOS>", "<UNK>"]

print(f"BPE 训练后的词表大小: {len(bpe.vocab)}")
print()

# 模拟：在词表后面追加特殊符号
extended_vocab = bpe.vocab.copy()
for i, tok in enumerate(special_tokens):
    extended_vocab[tok] = len(bpe.vocab) + i

print("扩展后的词表（最后几个）:")
for tok in special_tokens:
    print(f"  '{tok}' → ID {extended_vocab[tok]}")

print()
print("关键：这些 ID 是手动分配的，不是 BPE 训练出来的。")
print("训练时，模型会为这些特殊 token 也学习对应的 embedding 向量。")
print()

# 演示 PAD 的作用
print("=== PAD 的作用演示 ===")
print()
print("假设 batch 里有 3 句话，长度分别是 5, 3, 4:")
sentences = [
    [1, 2, 3, 4, 5],           # 长度 5
    [1, 2, 3],                 # 长度 3
    [1, 2, 3, 4],              # 长度 4
]

PAD_ID = extended_vocab["<PAD>"]
max_len = max(len(s) for s in sentences)

print(f"最长句子长度: {max_len}")
print(f"PAD_ID = {PAD_ID}")
print()

padded = []
for i, s in enumerate(sentences):
    pad_count = max_len - len(s)
    padded_s = s + [PAD_ID] * pad_count
    padded.append(padded_s)
    print(f"句子{i+1}: {s} → 补 {pad_count} 个 PAD → {padded_s}")

print()
print("现在 3 句话长度一致，可以组成一个 tensor 送入 GPU 并行计算了！")
print("（attention mask 会告诉模型忽略 PAD 位置，不算入注意力）")

### 13. 一个彩蛋：可视化 merge rules 的形成过程

把 merge rules 画出来，看看一个常见词是怎么被一步步「拼」出来的。

In [ ]:
print("Merge Rules 完整列表（按训练顺序）:")
print("注意看 'the' 是怎么诞生的：")
print()

for i, (a, b) in enumerate(bpe.merge_rules):
    arrow = ""
    # 标记和 'the' 相关的合并
    merged = a + b
    if merged in ['th', 'he', 'the']:
        arrow = "  ← 'the' 的诞生过程！"
    if a in ['th', 'he'] or b in ['th', 'he']:
        arrow = "  ← 'the' 的诞生过程！"
    print(f"  Rule {i+1:2d}: '{a}' + '{b}' → '{merged}'{arrow}")

print()
print("你看：'the' 不是一次性识别出来的，是 't'+'h'→'th'，然后合并能跟 'th' 组 pair 的字符，逐渐形成的。")

---

## 第二部分小结

确认你现在理解了这些：

1. ✅ BPE 从字符级开始，每一步合并最高频的相邻 pair
2. ✅ merge rules 是**有序的**，encode 时按这个顺序贪心应用
3. ✅ `num_merges` 控制词表大小，越大词表越大，token 越「整」
4. ✅ encode 的本质：反复应用 merge rules，把文本切成尽量大的 token
5. ✅ decode 的本质：查表 + 拼接，零难度
6. ✅ BPE 不崩的秘诀：遇到生词退回到字符级编码（反正字符一定在词表里）

**这些概念适用所有子词级 tokenizer**：WordPiece（BERT）、SentencePiece、Unigram… 都是「合并」或「拆分」的变体，核心数据结构都是 merge rules / vocab。

---

下一步 → **Part 3**：token ID 怎么变成神经网络能处理的向量？Embedding + Position Encoding！